# Objectif 1 - Application du pipeline IceTag McGill

Ce notebook applique le pipeline gele du memoire aux donnees IceTag McGill.

Principe:
- convertir les donnees IceTag minute par minute vers le format standard du pipeline en bins de 15 minutes;
- executer le pipeline `Isolation Forest + regles metier` sans modifier ses seuils;
- exporter les resumes, predictions et alertes.

A executer d'abord sur Fall 2019. Les autres saisons seront ajoutees apres inspection des fichiers exacts.

In [76]:
from pathlib import Path
import sys
import json
import datetime
import re
from collections import defaultdict

import pandas as pd
import numpy as np
import openpyxl

PROJECT = Path("/Users/alioubarry/PROJECT")
MCGILL = PROJECT / "mcgill_iot_cattle"
DATA_ROOT = MCGILL / "Données completes" / "Données accelerometres"
REPORTS = MCGILL / "reports" / "objective1_pipeline_icetag"
REPORTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT))

from core.io import normalize_columns
from core.pipeline import run_pipeline_herd

THRESHOLDS_PATH = PROJECT / "data" / "final_thresholds_v1.json"

with open(THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    thresholds = json.load(f)

PARAMS = thresholds["pipeline_defaults"].copy()
PARAMS.pop("sensor_warmup_bins", None)

PARAMS

{'interval': '15T',
 'window_baseline': 24,
 'contamination': 0.06,
 'baseline_ratio': 0.6,
 'random_state': 42,
 'persist_hours': 7,
 'alert_min': 2,
 'mix_mode': 'MIX',
 'mix_rate_thr': 0.24,
 'z_low_thr': -2.0,
 'z_high_thr': 2.0,
 'cooldown_hours': 12,
 'mi_z_high_thr': 2.2,
 'coverage_min_pct': 25.0}

## 1. Fichiers IceTag a traiter

On commence par Fall 2019, car ce fichier est deja identifie et compatible avec la conversion.

In [78]:
EXCEL_ICETAG_SOURCES = {
    "fall_2019": DATA_ROOT / "Fall 2019" / "Icetag" / "IceTags" / "@IceTag_Compiled_Lameness.xlsx",
}

CSV_ICETAG_SOURCES = {
    "summer_2019": DATA_ROOT / "Summer 2019" / "Icetag" / "IceTags" / "Per minute",
    "winter_2019": DATA_ROOT / "Winter 2019" / "Icetag" / "IceTags_Data" / "1 week IceTags (per minute)",
    "fall_2021": DATA_ROOT / "Fall 2021" / "Icetag" / "IceTags" / "IceTag Files - Fall 2021 - 06DEC2021",
}

FALL_2021_NOTES = DATA_ROOT / "Fall 2021" / "Icetag" / "IceTags" / "Copy of Ice tag download Notes1.xlsx"

print("Sources Excel compilees:")
for name, path in EXCEL_ICETAG_SOURCES.items():
    print(name, path.exists(), path)

print("\nSources CSV minute par minute:")
for name, path in CSV_ICETAG_SOURCES.items():
    n_csv = len(list(path.rglob("*.csv"))) if path.exists() else 0
    n_main_csv = len([p for p in path.rglob("*.csv") if not p.name.endswith("_LB.csv")]) if path.exists() else 0
    print(name, path.exists(), f"csv={n_csv}", f"main_csv={n_main_csv}", path)

print("\nNotes Fall 2021:", FALL_2021_NOTES.exists(), FALL_2021_NOTES)


Sources Excel compilees:
fall_2019 True /Users/alioubarry/PROJECT/mcgill_iot_cattle/Données completes/Données accelerometres/Fall 2019/Icetag/IceTags/@IceTag_Compiled_Lameness.xlsx

Sources CSV minute par minute:
summer_2019 True csv=377 main_csv=188 /Users/alioubarry/PROJECT/mcgill_iot_cattle/Données completes/Données accelerometres/Summer 2019/Icetag/IceTags/Per minute
winter_2019 True csv=470 main_csv=235 /Users/alioubarry/PROJECT/mcgill_iot_cattle/Données completes/Données accelerometres/Winter 2019/Icetag/IceTags_Data/1 week IceTags (per minute)
fall_2021 True csv=20 main_csv=10 /Users/alioubarry/PROJECT/mcgill_iot_cattle/Données completes/Données accelerometres/Fall 2021/Icetag/IceTags/IceTag Files - Fall 2021 - 06DEC2021

Notes Fall 2021: True /Users/alioubarry/PROJECT/mcgill_iot_cattle/Données completes/Données accelerometres/Fall 2021/Icetag/IceTags/Copy of Ice tag download Notes1.xlsx


## 2. Fonctions de conversion IceTag vers format pipeline

In [82]:
BIN_MINUTES = 15


def time_to_seconds(x):
    if x is None or pd.isna(x):
        return 0
    if isinstance(x, datetime.timedelta):
        return int(x.total_seconds())
    if isinstance(x, datetime.time):
        return x.hour * 3600 + x.minute * 60 + x.second
    if isinstance(x, datetime.datetime):
        t = x.time()
        return t.hour * 3600 + t.minute * 60 + t.second
    if isinstance(x, str):
        v = x.strip()
        try:
            parts = v.split(":")
            if len(parts) == 3:
                h, m, s = map(float, parts)
                return int(h * 3600 + m * 60 + s)
            if len(parts) == 2:
                m, s = map(float, parts)
                return int(m * 60 + s)
            return int(float(v))
        except Exception:
            return 0
    return 0


def seconds_to_hms(total_seconds):
    total_seconds = int(total_seconds)
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h}:{m:02d}:{s:02d}"


def make_bin_key(dt):
    minute_bin = (dt.minute // BIN_MINUTES) * BIN_MINUTES
    return dt.replace(minute=minute_bin, second=0, microsecond=0)


def normalize_colname(c):
    return str(c).strip().lower().replace("_", " ")


def find_col(columns, candidates):
    normalized = {normalize_colname(c): c for c in columns}
    for cand in candidates:
        key = normalize_colname(cand)
        if key in normalized:
            return normalized[key]
    return None


def aggregate_minutes_to_pipeline_rows(minutes, min_minutes_per_bin=10):
    bins = defaultdict(lambda: {
        "mi": 0,
        "standing_s": 0,
        "lying_s": 0,
        "steps": 0,
        "lb": 0,
        "count": 0,
    })

    for m in minutes:
        key = (m["Cow"], make_bin_key(m["dt"]))
        b = bins[key]
        b["mi"] += m["mi"]
        b["standing_s"] += m["standing_s"]
        b["lying_s"] += m["lying_s"]
        b["steps"] += m["steps"]
        b["lb"] += m["lb"]
        b["count"] += 1

    rows = []
    for (cow_id, bin_start), b in sorted(bins.items()):
        if b["count"] < min_minutes_per_bin:
            continue

        transitions = b["lb"]
        rows.append({
            "Cow": str(cow_id),
            "Start": bin_start.strftime("%Y-%m-%d %H:%M:%S"),
            "End": (bin_start + datetime.timedelta(minutes=BIN_MINUTES)).strftime("%Y-%m-%d %H:%M:%S"),
            "Steps": b["steps"],
            "Motion Index": b["mi"],
            "Lying Time": seconds_to_hms(b["lying_s"]),
            "Standing Time": seconds_to_hms(b["standing_s"]),
            "Transitions": transitions,
            "Transitions Up": transitions // 2,
            "Transitions Down": transitions - (transitions // 2),
        })

    return rows


def summarize_converted(out):
    if out.empty:
        return pd.DataFrame(columns=["Cow", "n_bins", "first_start", "last_start", "total_steps"])
    return out.groupby("Cow").agg(
        n_bins=("Start", "size"),
        first_start=("Start", "min"),
        last_start=("Start", "max"),
        total_steps=("Steps", "sum"),
    ).reset_index()


def convert_icetag_workbook_to_pipeline_csv(input_xlsx, output_csv, min_minutes_per_bin=10):
    wb = openpyxl.load_workbook(input_xlsx, read_only=True, data_only=True)
    all_minutes = []

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = list(ws.iter_rows(values_only=True))
        if len(rows) < 2:
            continue

        header = list(rows[0])
        col_cow = find_col(header, ["Cow", "Cow_ID", "CowID"])
        col_date = find_col(header, ["Date"])
        col_time = find_col(header, ["Time"])
        col_mi = find_col(header, ["Motion Index", "Motion_Index"])
        col_standing = find_col(header, ["Standing [t]", "Standing Time", "Standing"])
        col_lying = find_col(header, ["Lying [t]", "Lying Time", "Lying"])
        col_steps = find_col(header, ["Steps", "Step", "Sum_Steps"])
        col_lb = find_col(header, ["Lying Bouts", "Lying_Bouts", "LB"])

        required = [col_date, col_time, col_mi, col_standing, col_lying, col_steps]
        if any(c is None for c in required):
            print(f"Feuille ignoree, colonnes insuffisantes: {sheet_name}")
            print(header)
            continue

        idx = {c: header.index(c) for c in header if c is not None}

        for r in rows[1:]:
            date_val = r[idx[col_date]]
            time_val = r[idx[col_time]]
            if date_val is None or time_val is None:
                continue

            if isinstance(date_val, datetime.datetime):
                d = date_val.date()
            elif isinstance(date_val, datetime.date):
                d = date_val
            else:
                continue

            if isinstance(time_val, datetime.datetime):
                t = time_val.time()
            elif isinstance(time_val, datetime.time):
                t = time_val
            else:
                continue

            cow_id = r[idx[col_cow]] if col_cow is not None else None
            if cow_id is None or str(cow_id).strip() == "":
                cow_id = sheet_name

            all_minutes.append({
                "Cow": str(cow_id).strip(),
                "dt": datetime.datetime.combine(d, t),
                "mi": int(r[idx[col_mi]] or 0),
                "standing_s": time_to_seconds(r[idx[col_standing]]),
                "lying_s": time_to_seconds(r[idx[col_lying]]),
                "steps": int(r[idx[col_steps]] or 0),
                "lb": int(r[idx[col_lb]] or 0) if col_lb is not None else 0,
            })

    wb.close()
    out = pd.DataFrame(aggregate_minutes_to_pipeline_rows(all_minutes, min_minutes_per_bin=min_minutes_per_bin))
    out = out.sort_values(["Cow", "Start"]).reset_index(drop=True)
    out.to_csv(output_csv, index=False)
    return out, summarize_converted(out)


def cow_id_from_filename(path):
    name = Path(path).name
    match = re.search(r"(\d+)_Tag", name)
    return match.group(1) if match else None


def load_fall2021_tag_to_cow(notes_path):
    notes = pd.read_excel(notes_path)
    mapping = {}
    for _, row in notes.iterrows():
        if pd.notna(row.get("IceTag#")) and pd.notna(row.get("Cow_ID")):
            mapping[int(row["IceTag#"])] = str(int(row["Cow_ID"]))
    return mapping


def cow_id_from_fall2021_filename(path, tag_to_cow):
    match = re.search(r"IceTag\s+(\d+)", Path(path).name)
    if not match:
        return None
    tag_full = int(match.group(1))
    for candidate in (tag_full, tag_full % 1000, tag_full % 100):
        if candidate in tag_to_cow:
            return tag_to_cow[candidate]
    return None


def file_date_bounds(path):
    tokens = re.findall(r"(\d{2}[A-Za-z]{3}\d{4})", str(path))
    dates = []
    for token in tokens:
        try:
            dates.append(pd.to_datetime(token, format="%d%b%Y"))
        except Exception:
            pass
    if not dates:
        return None, None
    return min(dates), max(dates)


def parse_datetimes_with_file_context(date_series, time_series, csv_path):
    raw = date_series.astype(str) + " " + time_series.astype(str)
    candidates = [
        pd.to_datetime(raw, errors="coerce", dayfirst=False),
        pd.to_datetime(raw, errors="coerce", dayfirst=True),
    ]
    start_hint, end_hint = file_date_bounds(csv_path)
    if start_hint is None:
        return candidates[0]

    lower = start_hint - pd.Timedelta(days=2)
    upper = (end_hint if end_hint is not None else start_hint) + pd.Timedelta(days=2)

    def score(parsed):
        valid = parsed.notna()
        if valid.sum() == 0:
            return -1
        in_window = ((parsed >= lower) & (parsed <= upper)).sum()
        return int(in_window)

    return max(candidates, key=score)


def safe_int(value):
    parsed = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    if pd.isna(parsed):
        return 0
    return int(parsed)


def read_icetag_csv_minutes(csv_path, cow_id):
    df = pd.read_csv(csv_path, skiprows=7)
    required = ["Date", "Time", "Motion Index", "Standing [t]", "Lying [t]", "Steps", "Lying Bouts"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes dans {csv_path}: {missing}")

    dt = parse_datetimes_with_file_context(df["Date"], df["Time"], csv_path)
    valid = dt.notna()
    df = df.loc[valid].copy()
    dt = dt.loc[valid]

    return [
        {
            "Cow": str(cow_id),
            "dt": t.to_pydatetime(),
            "mi": safe_int(mi),
            "standing_s": time_to_seconds(standing),
            "lying_s": time_to_seconds(lying),
            "steps": safe_int(steps),
            "lb": safe_int(lb),
        }
        for t, mi, standing, lying, steps, lb in zip(
            dt,
            df["Motion Index"],
            df["Standing [t]"],
            df["Lying [t]"],
            df["Steps"],
            df["Lying Bouts"],
        )
    ]


def convert_icetag_csv_folder_to_pipeline_csv(season, input_dir, output_csv, min_minutes_per_bin=10):
    input_dir = Path(input_dir)
    csv_files = sorted(p for p in input_dir.rglob("*.csv") if not p.name.endswith("_LB.csv"))
    tag_to_cow = load_fall2021_tag_to_cow(FALL_2021_NOTES) if season == "fall_2021" else {}

    all_minutes = []
    skipped = []

    for csv_path in csv_files:
        cow_id = cow_id_from_fall2021_filename(csv_path, tag_to_cow) if season == "fall_2021" else cow_id_from_filename(csv_path)
        if cow_id is None:
            skipped.append((str(csv_path), "cow_id_introuvable"))
            continue
        try:
            all_minutes.extend(read_icetag_csv_minutes(csv_path, cow_id))
        except Exception as exc:
            skipped.append((str(csv_path), str(exc)))

    out = pd.DataFrame(aggregate_minutes_to_pipeline_rows(all_minutes, min_minutes_per_bin=min_minutes_per_bin))
    if not out.empty:
        out = out.drop_duplicates(["Cow", "Start"], keep="first")
        out = out.sort_values(["Cow", "Start"]).reset_index(drop=True)
    out.to_csv(output_csv, index=False)

    skipped_path = output_csv.with_name(output_csv.stem + "_skipped_files.csv")
    pd.DataFrame(skipped, columns=["file", "reason"]).to_csv(skipped_path, index=False)

    print(f"{season}: fichiers CSV lus={len(csv_files)}, ignores={len(skipped)}, minutes={len(all_minutes)}, bins={len(out)}")
    if skipped:
        print(f"Fichiers ignores: {skipped_path}")

    return out, summarize_converted(out)


## 3. Conversion des donnees

In [87]:
converted_paths = {}

for season, input_path in EXCEL_ICETAG_SOURCES.items():
    output_csv = REPORTS / f"{season}_pipeline_input_15min.csv"
    df_conv, df_conv_summary = convert_icetag_workbook_to_pipeline_csv(input_path, output_csv)
    converted_paths[season] = output_csv

    print("\n", "=" * 80)
    print(season)
    print("CSV:", output_csv)
    print("lignes:", len(df_conv))
    print("vaches:", df_conv["Cow"].nunique() if not df_conv.empty else 0)
    display(df_conv_summary.head())

for season, input_dir in CSV_ICETAG_SOURCES.items():
    output_csv = REPORTS / f"{season}_pipeline_input_15min.csv"
    df_conv, df_conv_summary = convert_icetag_csv_folder_to_pipeline_csv(season, input_dir, output_csv)
    converted_paths[season] = output_csv

    print("\n", "=" * 80)
    print(season)
    print("CSV:", output_csv)
    print("lignes:", len(df_conv))
    print("vaches:", df_conv["Cow"].nunique() if not df_conv.empty else 0)
    display(df_conv_summary.head())



fall_2019
CSV: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2019_pipeline_input_15min.csv
lignes: 93187
vaches: 30


,Cow,n_bins,first_start,last_start,total_steps
0,2041,3145,2019-11-11 18:45:00,2019-12-14 12:45:00,23143
1,2057,3144,2019-11-11 18:45:00,2019-12-14 12:30:00,19670
2,2062,3144,2019-11-11 18:45:00,2019-12-14 12:30:00,14685
3,2063,3140,2019-11-11 18:45:00,2019-12-14 11:30:00,19873
4,2066,3142,2019-11-11 18:45:00,2019-12-14 12:00:00,24302


/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %m/%d/%Y %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(raw, errors="coerce", dayfirst=True),
/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %m/%d/%Y %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(raw, errors="coerce", dayfirst=True),
/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %m/%d/%Y %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(raw, errors="coerce", dayfirst=True),
/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %

summer_2019: fichiers CSV lus=188, ignores=0, minutes=2053361, bins=136726

summer_2019
CSV: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/summer_2019_pipeline_input_15min.csv
lignes: 136726
vaches: 18


,Cow,n_bins,first_start,last_start,total_steps
0,2062,8922,2019-06-05 13:00:00,2019-09-06 11:15:00,56206
1,2067,1441,2019-06-05 13:00:00,2019-06-20 13:00:00,11064
2,419,8920,2019-06-05 13:00:00,2019-09-06 10:45:00,119397
3,5169,4803,2019-06-05 13:00:00,2019-07-25 13:30:00,34817
4,5214,1923,2019-07-05 08:30:00,2019-07-25 09:00:00,27062


...

winter_2019: fichiers CSV lus=235, ignores=16, minutes=10700717, bins=129323
Fichiers ignores: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/winter_2019_pipeline_input_15min_skipped_files.csv

winter_2019
CSV: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/winter_2019_pipeline_input_15min.csv
lignes: 129323
vaches: 17


,Cow,n_bins,first_start,last_start,total_steps
0,2047,6215,2019-01-16 15:15:00,2019-03-22 09:45:00,64944
1,2056,8714,2019-01-16 15:15:00,2019-04-17 10:30:00,59204
2,2063,8714,2019-01-16 15:15:00,2019-04-17 10:30:00,48941
3,2069,8713,2019-01-16 15:15:00,2019-04-17 10:15:00,72828
4,2081,8718,2019-01-16 15:15:00,2019-04-17 11:30:00,56715


/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %m/%d/%Y %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(raw, errors="coerce", dayfirst=True),
/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %m/%d/%Y %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(raw, errors="coerce", dayfirst=True),
/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %m/%d/%Y %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(raw, errors="coerce", dayfirst=True),
/var/folders/fr/5kl7mc5n6kn6xl137_xvbs6h0000gn/T/ipykernel_30524/2520621789.py:221: UserWarning: Parsing dates in %

fall_2021: fichiers CSV lus=10, ignores=0, minutes=77020, bins=5131

fall_2021
CSV: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2021_pipeline_input_15min.csv
lignes: 5131
vaches: 10


,Cow,n_bins,first_start,last_start,total_steps
0,2041,582,2021-11-30 09:30:00,2021-12-06 10:45:00,3838
1,2208,583,2021-11-30 09:30:00,2021-12-06 11:00:00,2445
2,8056,584,2021-11-30 09:15:00,2021-12-06 11:00:00,4001
3,8057,583,2021-11-30 09:30:00,2021-12-06 11:00:00,5823
4,8080,585,2021-11-30 09:00:00,2021-12-06 11:00:00,4980


## 4. Execution du pipeline gele

In [90]:
pipeline_outputs = {}

for season, csv_path in converted_paths.items():
    df = pd.read_csv(csv_path)
    df = normalize_columns(df)
    df["Cow"] = df["Cow"].astype(str)

    print(season, df.shape, df.columns.tolist()[:12])

    summary_df, predictions_df = run_pipeline_herd(df, **PARAMS)

    summary_path = REPORTS / f"{season}_pipeline_summary.csv"
    predictions_path = REPORTS / f"{season}_pipeline_predictions.csv"

    summary_df.to_csv(summary_path, index=False)
    predictions_df.to_csv(predictions_path, index=False)

    pipeline_outputs[season] = {
        "summary": summary_path,
        "predictions": predictions_path,
    }

    print("\n", "=" * 80)
    print(season)
    print("summary:", summary_path)
    print("predictions:", predictions_path)
    display(summary_df.head(10))

fall_2019 (93187, 10) ['Cow', 'T', 'End', 'Steps', 'Motion Index', 'Lying Time', 'Standing Time', 'Transitions', 'Transitions Up', 'Transitions Down']

fall_2019
summary: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2019_pipeline_summary.csv
predictions: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2019_pipeline_predictions.csv


,n_bins,if_anomaly_points,problem_points,lameness_points,problem_starts,lameness_starts,lameness_notifs,critique_points,coverage_mean,coverage_min,Cow
0,3144,193,78,78,10,10,8,17,100.0,100.0,2062
1,3072,206,92,92,8,8,8,13,100.0,100.0,8517
2,3141,172,72,72,7,7,7,16,100.0,100.0,5879
3,3142,222,34,34,7,7,6,5,100.0,100.0,2066
4,3144,182,47,47,6,6,6,7,100.0,100.0,5865
5,3144,219,74,74,7,7,5,4,100.0,100.0,5871
6,3141,177,47,47,6,6,5,7,100.0,100.0,2081
7,3145,166,56,56,6,6,5,11,100.0,100.0,2041
8,3072,232,69,69,5,5,5,7,100.0,100.0,8525
9,3072,182,34,34,5,5,5,8,100.0,100.0,8527


summer_2019 (136726, 10) ['Cow', 'T', 'End', 'Steps', 'Motion Index', 'Lying Time', 'Standing Time', 'Transitions', 'Transitions Up', 'Transitions Down']

summer_2019
summary: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/summer_2019_pipeline_summary.csv
predictions: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/summer_2019_pipeline_predictions.csv


,n_bins,if_anomaly_points,problem_points,lameness_points,problem_starts,lameness_starts,lameness_notifs,critique_points,coverage_mean,coverage_min,Cow
0,8922,582,224,224,18,18,17,19,100.000000,100.0,2062
1,7492,467,92,92,19,19,11,15,100.000000,100.0,5258
2,8920,549,89,89,13,13,11,13,100.000000,100.0,419
3,8921,477,47,47,15,15,10,8,100.000000,100.0,8536
4,8921,531,97,97,13,13,10,11,100.000000,100.0,5871
5,8920,503,53,53,12,12,10,10,100.000000,100.0,8520
6,8921,552,96,96,12,12,9,12,100.000000,100.0,5313
7,8920,564,58,58,10,10,9,4,97.724215,0.0,5857
8,8920,569,26,26,10,10,7,6,92.623318,0.0,5322
9,8925,526,74,74,10,10,7,7,100.000000,100.0,8515


winter_2019 (129323, 10) ['Cow', 'T', 'End', 'Steps', 'Motion Index', 'Lying Time', 'Standing Time', 'Transitions', 'Transitions Up', 'Transitions Down']

winter_2019
summary: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/winter_2019_pipeline_summary.csv
predictions: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/winter_2019_pipeline_predictions.csv


,n_bins,if_anomaly_points,problem_points,lameness_points,problem_starts,lameness_starts,lameness_notifs,critique_points,coverage_mean,coverage_min,Cow
0,8718,533,170,170,27,27,19,58,99.954118,0.0,8508
1,8718,404,144,144,22,22,17,41,99.954118,0.0,2056
2,8718,437,141,141,19,19,13,35,99.954118,0.0,5250
3,8718,427,110,110,19,19,12,44,92.349163,0.0,3443
4,8718,424,145,145,19,19,11,34,99.954118,0.0,8506
5,8722,432,148,148,15,15,11,35,99.954139,0.0,2081
6,6219,301,134,134,17,17,9,34,99.935681,0.0,2047
7,8718,395,114,114,11,11,9,39,92.268869,0.0,3437
8,8055,393,57,57,10,10,8,12,99.950341,0.0,5246
9,8718,427,98,98,9,9,8,20,99.954118,0.0,2063


fall_2021 (5131, 10) ['Cow', 'T', 'End', 'Steps', 'Motion Index', 'Lying Time', 'Standing Time', 'Transitions', 'Transitions Up', 'Transitions Down']

fall_2021
summary: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2021_pipeline_summary.csv
predictions: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2021_pipeline_predictions.csv


,n_bins,if_anomaly_points,problem_points,lameness_points,problem_starts,lameness_starts,lameness_notifs,critique_points,coverage_mean,coverage_min,Cow
0,457,24,15,15,2,2,1,3,100.0,100.0,8557
1,582,39,6,6,1,1,1,3,100.0,100.0,8562
2,584,37,7,7,1,1,1,1,100.0,100.0,8089
3,582,33,10,10,1,1,1,4,100.0,100.0,2041
4,583,37,0,0,0,0,0,0,100.0,100.0,8057
5,584,33,0,0,0,0,0,0,100.0,100.0,8056
6,585,33,0,0,0,0,0,0,100.0,100.0,8080
7,582,33,0,0,0,0,0,0,100.0,100.0,8537
8,583,32,0,0,0,0,0,0,100.0,100.0,2208
9,9,1,0,0,0,0,0,0,100.0,100.0,8539


## 5. Extraction des notifications uniquement

In [92]:
for season, paths in pipeline_outputs.items():
    pred = pd.read_csv(paths["predictions"])

    alert_cols = [
        "Cow", "T",
        "if_anomaly_point",
        "pred_problem_episode",
        "pred_problem_start",
        "pred_lameness_episode",
        "pred_lameness_start",
        "notif_lameness",
        "lame_confidence",
        "coverage_pct",
    ]

    existing_cols = [c for c in alert_cols if c in pred.columns]
    alerts = pred[pred.get("notif_lameness", 0) == 1][existing_cols].copy()

    alerts_path = REPORTS / f"{season}_pipeline_alerts_only.csv"
    alerts.to_csv(alerts_path, index=False)

    print("\n", "=" * 80)
    print(season)
    print("alertes:", len(alerts))
    print("fichier:", alerts_path)
    display(alerts.head(20))


fall_2019
alertes: 105
fichier: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2019_pipeline_alerts_only.csv


,Cow,T,if_anomaly_point,pred_problem_episode,pred_problem_start,pred_lameness_episode,pred_lameness_start,notif_lameness,lame_confidence,coverage_pct
70,2041,2019-11-12 12:15:00,1,1,1,1,1,1,47.1,100.0
343,2041,2019-11-15 08:30:00,0,1,1,1,1,1,42.4,100.0
642,2041,2019-11-18 11:15:00,1,1,1,1,1,1,58.7,100.0
933,2041,2019-11-21 12:00:00,1,1,1,1,1,1,48.4,100.0
1314,2041,2019-11-25 11:15:00,1,1,1,1,1,1,51.5,100.0
3213,2057,2019-11-12 11:45:00,1,1,1,1,1,1,50.0,100.0
4291,2057,2019-11-23 17:15:00,1,1,1,1,1,1,47.1,100.0
4383,2057,2019-11-24 16:15:00,1,1,1,1,1,1,47.0,100.0
4747,2057,2019-11-28 11:15:00,1,1,1,1,1,1,50.2,100.0
6917,2062,2019-11-18 07:45:00,0,1,1,1,1,1,45.4,100.0



summer_2019
alertes: 127
fichier: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/summer_2019_pipeline_alerts_only.csv


,Cow,T,if_anomaly_point,pred_problem_episode,pred_problem_start,pred_lameness_episode,pred_lameness_start,notif_lameness,lame_confidence,coverage_pct
369,2062,2019-06-09 09:15:00,1,1,1,1,1,1,46.2,100.0
649,2062,2019-06-12 07:15:00,1,1,1,1,1,1,50.1,100.0
848,2062,2019-06-14 09:00:00,1,1,1,1,1,1,47.5,100.0
1512,2062,2019-06-21 07:00:00,0,1,1,1,1,1,45.8,100.0
1813,2062,2019-06-24 10:15:00,1,1,1,1,1,1,56.5,100.0
1997,2062,2019-06-26 08:15:00,1,1,1,1,1,1,46.7,100.0
4585,2062,2019-07-23 07:15:00,0,1,1,1,1,1,40.8,100.0
4872,2062,2019-07-26 07:00:00,0,1,1,1,1,1,42.9,100.0
6802,2062,2019-08-15 09:30:00,0,1,1,1,1,1,48.0,100.0
7376,2062,2019-08-21 09:00:00,1,1,1,1,1,1,47.2,100.0



winter_2019
alertes: 149
fichier: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/winter_2019_pipeline_alerts_only.csv


,Cow,T,if_anomaly_point,pred_problem_episode,pred_problem_start,pred_lameness_episode,pred_lameness_start,notif_lameness,lame_confidence,coverage_pct
1520,2047,2019-02-01 11:15:00,1,1,1,1,1,1,56.8,100.0
1607,2047,2019-02-02 09:00:00,0,1,1,1,1,1,47.1,100.0
1713,2047,2019-02-03 11:30:00,0,1,1,1,1,1,44.6,100.0
1839,2047,2019-02-04 19:00:00,1,1,1,1,1,1,46.8,100.0
1893,2047,2019-02-05 08:30:00,1,1,1,1,1,1,48.3,100.0
1985,2047,2019-02-06 07:30:00,0,1,1,1,1,1,43.1,100.0
2040,2047,2019-02-06 21:15:00,0,1,1,1,1,1,48.6,100.0
2092,2047,2019-02-07 10:15:00,1,1,1,1,1,1,49.4,100.0
2187,2047,2019-02-08 10:00:00,1,1,1,1,1,1,49.1,100.0
7991,2056,2019-02-04 02:15:00,1,1,1,1,1,1,51.0,100.0



fall_2021
alertes: 4
fichier: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2021_pipeline_alerts_only.csv


,Cow,T,if_anomaly_point,pred_problem_episode,pred_problem_start,pred_lameness_episode,pred_lameness_start,notif_lameness,lame_confidence,coverage_pct
99,2041,2021-12-01 10:15:00,1,1,1,1,1,1,48.1,100.0
3217,8089,2021-12-03 12:15:00,1,1,1,1,1,1,49.4,100.0
4197,8557,2021-12-01 11:45:00,1,1,1,1,1,1,57.3,100.0
4750,8562,2021-12-02 12:00:00,1,1,1,1,1,1,58.1,100.0


## 6. Synthese multi-saisons


In [96]:
season_rows = []

for season, paths in pipeline_outputs.items():
    summary = pd.read_csv(paths["summary"])
    pred = pd.read_csv(paths["predictions"])
    pred_t = pd.to_datetime(pred["T"], errors="coerce")
    cow_days = len(pred) / 96
    n_notifs = int(pred["notif_lameness"].sum()) if "notif_lameness" in pred else 0

    season_rows.append({
        "season": season,
        "n_cows": int(summary["Cow"].nunique()),
        "n_intervals": int(len(pred)),
        "first_T": pred_t.min(),
        "last_T": pred_t.max(),
        "if_anomaly_points": int(pred["if_anomaly_point"].sum()),
        "lameness_points": int(pred["pred_lameness_episode"].sum()),
        "lameness_starts": int(pred["pred_lameness_start"].sum()),
        "lameness_notifs": n_notifs,
        "notifs_per_100_cow_days": round(n_notifs / cow_days * 100, 2) if cow_days else np.nan,
        "min_coverage": float(pred["coverage_pct"].min()),
        "mean_coverage": float(pred["coverage_pct"].mean()),
    })

multi_season_summary = pd.DataFrame(season_rows).sort_values("season")
multi_summary_path = REPORTS / "objective1_multi_season_summary.csv"
multi_season_summary.to_csv(multi_summary_path, index=False)

print(multi_summary_path)
display(multi_season_summary)


/Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/objective1_multi_season_summary.csv


,season,n_cows,n_intervals,first_T,last_T,if_anomaly_points,lameness_points,lameness_starts,lameness_notifs,notifs_per_100_cow_days,min_coverage,mean_coverage
0,fall_2019,30,93860,2019-11-11 18:45:00,2019-12-14 13:00:00,5711,1104,121,105,10.74,0.0,99.282975
3,fall_2021,10,5131,2021-11-30 09:00:00,2021-12-06 11:00:00,302,38,5,4,7.48,100.0,100.000000
1,summer_2019,18,139111,2019-06-05 13:00:00,2019-09-06 12:00:00,8081,1045,166,127,8.76,0.0,98.285542
2,winter_2019,17,136929,2019-01-16 15:15:00,2019-04-17 11:30:00,6462,1635,214,149,10.45,0.0,94.445296


## 7. Note technique de reproductibilité

In [99]:
lines = []
lines.append("# Note technique - Objectif 1")
lines.append("")
lines.append("Pipeline applique: Isolation Forest + regles metier.")
lines.append(f"Seuils geles: {THRESHOLDS_PATH}")
lines.append("")
lines.append("Parametres utilises:")
for k, v in PARAMS.items():
    lines.append(f"- {k}: {v}")

lines.append("")
lines.append("Sorties generees:")

for season, paths in pipeline_outputs.items():
    summary = pd.read_csv(paths["summary"])
    pred = pd.read_csv(paths["predictions"])

    n_alerts = int(pred["notif_lameness"].sum()) if "notif_lameness" in pred else 0

    lines.append("")
    lines.append(f"## {season}")
    lines.append(f"- vaches: {summary['Cow'].nunique()}")
    lines.append(f"- intervalles predits: {len(pred)}")
    lines.append(f"- notifications boiterie: {n_alerts}")
    lines.append(f"- resume: {paths['summary']}")
    lines.append(f"- predictions: {paths['predictions']}")

if "multi_season_summary" in globals():
    lines.append("")
    lines.append("## Synthese multi-saisons")
    for _, row in multi_season_summary.iterrows():
        lines.append(f"- {row['season']}: {int(row['n_cows'])} vaches, {int(row['n_intervals'])} intervalles, {int(row['lameness_notifs'])} notifications, {row['notifs_per_100_cow_days']} notifications / 100 vache-jours")

note_path = REPORTS / "objective1_technical_note.md"
note_path.write_text("\n".join(lines), encoding="utf-8")

print(note_path)
print(note_path.read_text(encoding="utf-8"))

/Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/objective1_technical_note.md
# Note technique - Objectif 1

Pipeline applique: Isolation Forest + regles metier.
Seuils geles: /Users/alioubarry/PROJECT/data/final_thresholds_v1.json

Parametres utilises:
- interval: 15T
- window_baseline: 24
- contamination: 0.06
- baseline_ratio: 0.6
- random_state: 42
- persist_hours: 7
- alert_min: 2
- mix_mode: MIX
- mix_rate_thr: 0.24
- z_low_thr: -2.0
- z_high_thr: 2.0
- cooldown_hours: 12
- mi_z_high_thr: 2.2
- coverage_min_pct: 25.0

Sorties generees:

## fall_2019
- vaches: 30
- intervalles predits: 93860
- notifications boiterie: 105
- resume: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2019_pipeline_summary.csv
- predictions: /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective1_pipeline_icetag/fall_2019_pipeline_predictions.csv

## summer_2019
- vaches: 18
- intervalles predits: 139111
- notifications boiterie: 12